Project: Implementing a Chatgpt App With Langchain From Scratch

In [1]:
from dotenv import load_dotenv, find_dotenv
load_dotenv(find_dotenv(),override=True)

True

In [3]:
from langchain_openai import ChatOpenAI
from langchain_core.messages import SystemMessage
from langchain_core.prompts import ChatPromptTemplate,HumanMessagePromptTemplate
from langchain_core.output_parsers import StrOutputParser

llm=ChatOpenAI(model_name='gpt-3.5-turbo',temperature=1)
prompt=ChatPromptTemplate(
    input_variable=['content'],
    messages=[SystemMessage(content="you are a chatbot having conversation with a human"),
              HumanMessagePromptTemplate.from_template("{content}")
              ]
)

chain=prompt | llm | StrOutputParser()
while True:
    content=input("Your Prompt.....")
    if content.lower() in ['bye','ok','exit']:
        print("Goodbye")
        break

    response=chain.invoke({'content':content})
    print(response)
    print("-"*50)

Goodbye


Adding Chat Memory using ConversationBufferMemory

In [4]:
from langchain_openai import ChatOpenAI
from langchain_core.messages import SystemMessage
from langchain_core.prompts import ChatPromptTemplate, HumanMessagePromptTemplate, MessagesPlaceholder
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables.history import RunnableWithMessageHistory
from langchain_core.chat_history import InMemoryChatMessageHistory

llm = ChatOpenAI(model_name="gpt-3.5-turbo", temperature=0.7)

store = {}

def get_session_history(session_id: str):
    if session_id not in store:
        store[session_id] = InMemoryChatMessageHistory()
    return store[session_id]

# Fixed Prompt Section
prompt = ChatPromptTemplate.from_messages([
    SystemMessage(content="You are a chatbot having a conversation with a human."),
    MessagesPlaceholder("chat_history"),
    HumanMessagePromptTemplate.from_template("{content}")
])

chain = RunnableWithMessageHistory(
    prompt | llm | StrOutputParser(),
    get_session_history,
    input_messages_key="content",
    history_messages_key="chat_history"
)

while True:
    content = input("You Prompt....... ")
    if content.lower() in ["goodbye", "bye", "end"]:
        print("Signing off!")
        break

    response = chain.invoke(
        {"content": content},
        config={"configurable": {"session_id": "default"}}
    )
    
    print(response)
    print("-" * 50)

C:\Users\Mr. Tech\AppData\Roaming\Python\Python312\site-packages\IPython\core\interactiveshell.py:3748: LangChainPendingDeprecationWarning: RunnableWithMessageHistory is deprecated. Use LangGraph's built-in persistence instead.
  exec(code_obj, self.user_global_ns, self.user_ns)


Signing off!
